# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I frame this problem as a **ranking task** because the main decision is not simply whether a page is good or bad. The goal is to rank pages by their review priority so a content team can focus on the pages that appear most worth reviewing first. Multiple signals such as impressions, sessions, content age, position, CTR, and recent trend can contribute to this priority. The output is therefore an ordered list rather than a simple yes/no prediction.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the dataset is available and show the basic unit we will rank

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())

Dataset shape: (30000, 44)
Unique content IDs: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is a proxy for **pages showing a decline signal that may deserve review**. I will use the available `trend_direction` field as the starting label, with pages marked as `down` treated as the positive class for this framing.

This is a proxy rather than a true business outcome. It tells us that a page shows an observed downward trend, but it does not tell us that a refresh would have caused the page to recover. The model should therefore be used to prioritize pages for human review, not to claim that an intervention will succeed.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect the target/proxy distribution

target_counts = df["trend_direction"].value_counts(dropna=False)

print("Trend direction counts:")
print(target_counts)

print("\nDown rate:")
print(f"{(df['trend_direction'] == 'down').mean():.2%}")

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Down rate:
54.21%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use **Precision@K** as the main success metric. K represents the number of pages the content team can realistically review in one batch.

Precision@K measures how many of the top K ranked pages are actually marked as `down`. This fits the decision because the team cares about getting useful pages into the limited review queue, rather than ranking every page equally well.

For example, Precision@50 asks: "Among the 50 pages we ranked highest, how many are actually marked as down?"

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple Precision@K function for our ranking task

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.mean(np.asarray(y_true)[order])

# Baseline: rank pages by a simple observable signal.
# Higher impressions are treated as higher priority for this example.
y_true = (df["trend_direction"] == "down").astype(int)
baseline_scores = df["impressions_90d"].to_numpy()

print("Precision@50 baseline:",
      round(precision_at_k(y_true, baseline_scores, 50), 3))

Precision@50 baseline: 0.42


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is **one content page**. Each row represents one page identified by `content_id`, with its client, search performance, engagement, content age, and trend information.

This matches the decision because the content team reviews and takes action on individual pages. The model will therefore produce a priority score for each page, which can then be used to create an ordered review queue.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the actual unit of analysis as a dataframe

page_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction"
]

page_df = df[page_columns].copy()

print("Rows:", len(page_df))
print("Columns:", list(page_df.columns))
print("\nOne row represents one content page:")
display(page_df.head())

Rows: 30000
Columns: ['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'content_age_days', 'trend_direction']

One row represents one content page:


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could rank pages using one signal, such as impressions or content age. However, page review priority may depend on several signals at the same time. An ML ranking model can learn how these signals work together instead of relying on one manually chosen threshold.

The benefit is not that ML guarantees better content decisions. The benefit is that it can learn a more flexible ranking from historical examples and then produce a consistent review queue. I will compare the ML approach with a simple baseline using Precision@K to check whether the extra complexity is useful.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compare the number of pages available for ranking
# with the number marked as down.

total_pages = len(df)
down_pages = (df["trend_direction"] == "down").sum()

print("Total pages available for ranking:", total_pages)
print("Pages marked as down:", down_pages)
print(f"Down rate: {down_pages / total_pages:.2%}")

Total pages available for ranking: 30000
Pages marked as down: 16262
Down rate: 54.21%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.